# Group2 Baseline Workflow (Orchestration)

This notebook keeps heavy logic in `src/group2_stage2` and only orchestrates steps.


## Stage 0 Checklist
- [ ] Group1 environment (`.venv`) is active
- [ ] `configs/workflow_paths.json` is updated
- [ ] Stage2 variant datasets exist under `stage2_root/<variant>/stage2_dataset.jsonl`


In [ ]:
import json
from pathlib import Path

project_root = Path("..").resolve()
config_path = project_root / "configs" / "workflow_paths.json"

raw_cfg = json.loads(config_path.read_text(encoding="utf-8"))
cfg = {
    k: (v.replace("${PROJECT_ROOT}", str(project_root)) if isinstance(v, str) else v)
    for k, v in raw_cfg.items()
}

stage2_root = Path(cfg["stage2_root"])
image_root = Path(cfg["image_root"])
feature_root = Path(cfg["clip_feature_root"])
all_variants = [cfg["baseline_variant"]] + cfg.get("quality_variants", [])

print("project_root:", project_root)
print("stage2_root:", stage2_root)
print("all_variants:", all_variants)


## Stage 1: Audit + Shared Split


In [ ]:
from src.group2_stage2.audit import audit_stage2_variants
from src.group2_stage2.splits import build_shared_quality_pool, materialize_train_val_split

audit = audit_stage2_variants(stage2_root, all_variants)
audit


In [ ]:
pool = build_shared_quality_pool(
    stage2_root=stage2_root,
    all_variants=all_variants,
    quality_image_count=int(cfg["quality_image_count"]),
    val_image_count=int(cfg["val_image_count"]),
    split_seed=int(cfg.get("split_seed", 42)),
    pool_reference_variant=cfg["baseline_variant"],
)
pool


In [ ]:
split_stats = materialize_train_val_split(stage2_root, all_variants)
split_stats


## Stage 2 Checklist (model-dependent)
- [ ] Group1 CLIP bundle loaded
- [ ] Group1 tokenizer loaded
- [ ] `get_features_compiled` ready from Group1 CLIP helper


In [ ]:
# Example imports from Group1 baseline runtime (same environment/session):
# from src.vision_features.clip_helpers import build_clip_vision_tower
# from src.training.clip_features import make_clip_feature_fn
# from transformers import AutoTokenizer
# clip_bundle = build_clip_vision_tower()
# get_features_compiled = make_clip_feature_fn(clip_bundle)
# tokenizer = AutoTokenizer.from_pretrained(...)


In [ ]:
from src.group2_stage2.pipeline import prepare_stage2_variant_splits

# Uncomment when tokenizer/clip_bundle/get_features_compiled are initialized:
# prep_results = prepare_stage2_variant_splits(
#     stage2_root=stage2_root,
#     image_root=image_root,
#     feature_root=feature_root,
#     tokenizer=tokenizer,
#     clip_bundle=clip_bundle,
#     get_features_compiled=get_features_compiled,
#     all_variants=all_variants,
# )
# prep_results


## Stage 3: Evaluation Artifacts


In [ ]:
from src.group2_stage2.quality_eval import (
    build_dataset_quality_diagnostics,
    build_qualitative_samples_pack,
    build_pairwise_judge_requests,
)

quality = build_dataset_quality_diagnostics(stage2_root, all_variants)
qual_samples = build_qualitative_samples_pack(stage2_root, all_variants)

# Requires heldout_eval_pack.json already prepared:
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"])


## Stage 4: Quality Experiment Tracking
Use these after stage2 manifests/training are available.


In [ ]:
from src.group2_stage2.experiment_tracking import (
    select_next_variant,
    run_and_store_variant,
    prompt_alignment_audit,
    build_engine_comparison_summary,
    build_baseline_relative_comparison,
)

results_path = stage2_root / "all_results_manual.json"
selection = select_next_variant(
    results_path=results_path,
    expected_variants=all_variants,
    current_variant=None,
    allow_overwrite=False,
)
selection


In [ ]:
# Wire your stage2 experiment runner here (model-dependent):
# run_and_store_variant(
#     results_path=results_path,
#     all_results=selection["all_results"],
#     selected_variant=selection["selected_variant"],
#     run_stage2_experiment_fn=run_stage2_experiment,
# )

prompt_alignment_audit(stage2_root, all_variants, prompt_reference_variant=all_variants[0])


In [ ]:
# Requires all expected variants in all_results_manual.json
# build_engine_comparison_summary(stage2_root, results_path, expected_variants=all_variants)
# build_baseline_relative_comparison(stage2_root, results_path, baseline_variant=cfg["baseline_variant"], expected_variants=all_variants)


## Stage 5: Quantity Ablation


In [ ]:
from src.group2_stage2.quantity_ablation import (
    derive_quantity_plan,
    build_quantity_variants,
    register_quantity_variants,
    prepare_quantity_variants,
    run_quantity_experiments,
    summarize_quantity_results,
)

plan = derive_quantity_plan(stage2_root)
plan


In [ ]:
quantity_variants = build_quantity_variants(
    stage2_root=stage2_root,
    quantity_source_variant=plan["quantity_source_variant"],
    quantity_levels=plan["quantity_levels"],
    quantity_split_seed=plan["quantity_split_seed"],
)
register_quantity_variants(stage2_root, quantity_variants)


In [ ]:
# model-dependent preparation + run
# prepare_quantity_variants(
#     stage2_root,
#     quantity_variants,
#     tokenize_fn=tokenize_stage2_variant,
#     extract_features_fn=extract_stage2_features,
#     build_manifest_fn=build_stage2_manifest,
# )
# run_quantity_experiments(stage2_root, quantity_variants, run_stage2_experiment_fn=run_stage2_experiment)
# summarize_quantity_results(stage2_root)


## Stage 6: Heldout Pack + Pairwise Requests


In [ ]:
from src.group2_stage2.evaluation_pack import build_heldout_eval_pack
from src.group2_stage2.quality_eval import build_pairwise_judge_requests

heldout = build_heldout_eval_pack(stage2_root, all_variants, samples_per_task=10, seed=123)
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"], seed=2026)
heldout["num_samples"]


## Stage 7: Reporting Figures


In [ ]:
from src.group2_stage2.reporting import build_engine_plots_and_table

# Requires engine summary + quality diagnostics json outputs.
# build_engine_plots_and_table(stage2_root)
